In [ ]:
import requests
import gzip
import json

# Updated URL — the dataset moved to a new domain
url = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz"

sample_reviews = []
max_reviews = 500

response = requests.get(url, stream=True)
with gzip.open(response.raw) as f:
    for i, line in enumerate(f):
        if i >= max_reviews:
            break
        review = json.loads(line)
        sample_reviews.append(review)

print(f"Collected {len(sample_reviews)} sample reviews")
print(sample_reviews[0])

Collected 500 sample reviews
{'user_id': '8842281e1d1347389f2ab93d60773d4d', 'book_id': '1893', 'review_id': '5347a776a1703b823ce029d68ae98275', 'rating': 5, 'review_text': "** spoiler alert ** \n So the other day Elizabeth and I are in the book store and she saw this book, and said she really wanted me to read it. In horror at the thought of reading what I thought was a 'chick book', i immediately countered that she would then have to read one of my favorites: Dune. She agreed! \n So I read it, and I have to admit, it was good - damn good. Even though there was a serious lack of any gratuitous violence, I tore through it in several days. Austen is an amazing writer, and has a particular talent for explaining her characters deep motivations (or prejudices) in a few defining sentences. \n I think my favorite part of it is the unwinding of Elizabeths' prejudices against Mr Darcy. It is done so slowly and artfully and believably that the reader is completely pulled into the story. \n It i

In [ ]:
import re

# Basic check: how many reviews are spoiler-tagged?
spoiler_count = sum(1 for r in sample_reviews if '** spoiler alert **' in r['review_text'].lower())
print(f"Spoiler-tagged reviews: {spoiler_count} out of {len(sample_reviews)}")

# Check review length distribution (word count)
lengths = [len(r['review_text'].split()) for r in sample_reviews]
print(f"Avg review length (words): {sum(lengths)/len(lengths):.1f}")
print(f"Shortest: {min(lengths)}, Longest: {max(lengths)}")

# Peek at a few very short reviews — often low quality/low signal
short_reviews = [r['review_text'] for r in sample_reviews if len(r['review_text'].split()) < 10]
print(f"\nNumber of very short reviews (<10 words): {len(short_reviews)}")
for r in short_reviews[:3]:
    print("-", r)

Spoiler-tagged reviews: 2 out of 500
Avg review length (words): 295.2
Shortest: 1, Longest: 862

Number of very short reviews (<10 words): 8
- Eh...a lil harlequin romance anyone?
- Absolutely enthralling!! Review to follow soon.
- Very rough, action-packed, and super-sexy! Review to follow.


In [ ]:
import re

def clean_review(text):
    text = text.replace('** spoiler alert **', '').strip()  # remove spoiler tag itself
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines into one
    return text

# Filter: remove very short/placeholder reviews, clean the rest
cleaned_reviews = []
for r in sample_reviews:
    word_count = len(r['review_text'].split())
    if word_count >= 15:   # drop placeholder/junk reviews
        r['review_text_clean'] = clean_review(r['review_text'])
        cleaned_reviews.append(r)

print(f"Kept {len(cleaned_reviews)} out of {len(sample_reviews)} reviews after cleaning")
print(cleaned_reviews[0]['review_text_clean'][:300])

Kept 489 out of 500 reviews after cleaning
So the other day Elizabeth and I are in the book store and she saw this book, and said she really wanted me to read it. In horror at the thought of reading what I thought was a 'chick book', i immediately countered that she would then have to read one of my favorites: Dune. She agreed! So I read it,


In [ ]:
import requests
import gzip
import json

url = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz"

target_count = 20000       # how many qualifying reviews we want
min_votes = 5               # engagement threshold
max_lines_to_scan = 500000  # safety cap so we don't run forever if threshold is too strict

collected = []
lines_scanned = 0

response = requests.get(url, stream=True)
with gzip.open(response.raw) as f:
    for line in f:
        lines_scanned += 1
        if lines_scanned > max_lines_to_scan or len(collected) >= target_count:
            break

        review = json.loads(line)

        # Apply both our filters: engagement AND minimum length
        if review['n_votes'] >= min_votes and len(review['review_text'].split()) >= 15:
            collected.append(review)

print(f"Scanned {lines_scanned} lines, collected {len(collected)} qualifying reviews")

Scanned 359252 lines, collected 20000 qualifying reviews


In [ ]:
def clean_review(text):
    text = text.replace('** spoiler alert **', '').strip()
    text = re.sub(r'\s+', ' ', text)
    return text

for review in collected:
    review['review_text_clean'] = clean_review(review['review_text'])

print(f"Cleaned {len(collected)} reviews")
print(collected[0]['review_text_clean'][:200])

Cleaned 20000 reviews
So the other day Elizabeth and I are in the book store and she saw this book, and said she really wanted me to read it. In horror at the thought of reading what I thought was a 'chick book', i immedia


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import os

# Create a project folder in your Drive
save_dir = '/content/drive/MyDrive/romance_vibe_project'
os.makedirs(save_dir, exist_ok=True)

save_path = f'{save_dir}/romance_reviews_clean.json'
with open(save_path, 'w') as f:
    json.dump(collected, f)

print(f"Saved {len(collected)} reviews to {save_path}")

Saved 20000 reviews to /content/drive/MyDrive/romance_vibe_project/romance_reviews_clean.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import json

load_path = '/content/drive/MyDrive/romance_vibe_project/romance_reviews_clean.json'
with open(load_path, 'r') as f:
    collected = json.load(f)

print(f"Loaded {len(collected)} reviews back into memory")

Mounted at /content/drive
Loaded 20000 reviews back into memory


In [3]:
!pip install vaderSentiment -q

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Test on a few reviews first
for review in collected[:5]:
    score = analyzer.polarity_scores(review['review_text_clean'])
    print(f"Rating: {review['rating']} | VADER compound: {score['compound']:.3f}")
    print(review['review_text_clean'][:150])
    print("---")

Rating: 5 | VADER compound: 0.853
So the other day Elizabeth and I are in the book store and she saw this book, and said she really wanted me to read it. In horror at the thought of re
---
Rating: 5 | VADER compound: 0.991
4.5 I'm-Obsessed Stars!! Jessica Hawkins has written an engrossing and emotional romance that I simply couldn't put down! I am obsessed with Lake and 
---
Rating: 5 | VADER compound: 0.997
4.5 Stars!! "Was sixteen too young to fall in love? I might've thought so before Manning. Could he love me back, a man seven years older? I was sure i
---
Rating: 5 | VADER compound: 0.996
4.5 EPIC STARS!! OMG what do I read now?! My heart is FILLED with Lake and Manning. A truly epic and passionate series. A must-read! "You wanted me to
---
Rating: 4 | VADER compound: 0.995
RACER EFFING TATE!! Katy Evans is back, doing what she does best! 'Real' fans will swoon because Remy and Brooke's little boy Racer is all grown up! T
---


In [4]:
# Search for reviews that mention "dark" to see how VADER handles ambiguous/intense language
dark_reviews = [r for r in collected if 'dark' in r['review_text_clean'].lower()][:5]

for review in dark_reviews:
    score = analyzer.polarity_scores(review['review_text_clean'])
    print(f"Rating: {review['rating']} | VADER compound: {score['compound']:.3f}")
    print(review['review_text_clean'][:200])
    print("---")

Rating: 4 | VADER compound: 0.996
4.5 Darkly Delicious Stars!!! Adult, sexy and daring. Brianna Hale gets better and better. "His singing voice can rattle windows with fury or caress with love." Ahhhhh....Frederic. Each hero of Briann
---
Rating: 5 | VADER compound: 0.932
"We got here, baby, through fire and blades, demons and curses, blood and plagues." Just WOW. I had high hopes for 'Fury' and Cat Porter exceeded every single one of my expectations and then some!! A 
---
Rating: 3 | VADER compound: 0.997
I ALWAYS love a Kitty Thomas book! This one is dark yet very intriguing and sexy. It can be read as a standalone. This story started off very promising for me. I was fascinated by Julie, who worked as
---
Rating: 4 | VADER compound: 0.997
Smart and sexy and delicious! I devoured this one. LOVE Ryder!!!! This is my second 1001 Dark nights novella and I really like this concept. Many of my favorite authors have written a novella in this 
---
Rating: 4 | VADER compound: 0.999
4.5 Stars!